# Your First dapi Job

One calculation, every dapi step. A 5%-damped oscillator is shaken exactly at its natural frequency, and one job on Stampede3 integrates the motion and reports the dynamic amplification, which lands at the top of the resonance curve, near $1/(2\xi) = 10$.

![A mass on a spring and damper, forced by a sine load, beside the amplification curve, with marked points.](resonance_schematic.png)

The oscillator obeys

$$\ddot{x} + 2\xi\omega_n\dot{x} + \omega_n^2 x = \frac{F_0}{m}\sin(\Omega t)$$

and this notebook computes the single point at $\Omega = \omega_n$. The [PyLauncher sweep](../pylauncher/pylauncher_sweep.ipynb) fills in the rest of the curve with one task per frequency; here the point is the dapi lifecycle itself, from a folder on disk to a result file in the archive.

In [ ]:
%pip install --quiet --upgrade dapi

**Restart the kernel once after the install**, then run from the next cell.

## Authenticate

`DSClient()` reads your DesignSafe credentials from the environment or a `.env` file, or prompts for them, and holds the authenticated Tapis connection every later call uses.

Docs: [Authentication](https://designsafe-ci.github.io/dapi/authentication).

In [1]:
from dapi import DSClient

ds = DSClient()

/Users/krishna/dev/DesignSafe/Dapi-Tapis/dapi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Authentication successful.


TMS credentials ready: frontera, stampede3


## Write the analysis

The script integrates the oscillator at resonance and writes one number to `result.json`. It lands in a folder that will become the job's input directory.

In [2]:
from pathlib import Path

input_dir = Path.cwd() / "oscillator_job"
input_dir.mkdir(exist_ok=True)

script = """\
# A 5%-damped oscillator shaken at its natural frequency.

import json

import numpy as np

xi = 0.05
wn = 1.0
w = 1.0 * wn                       # forcing at resonance

dt = 0.002
n = int(80 * 2 * np.pi / wn / dt)  # 80 natural periods
x = np.zeros(n)
t = np.arange(n) * dt
for i in range(1, n - 1):
    a = np.sin(w * t[i]) - 2 * xi * wn * (x[i] - x[i - 1]) / dt - wn**2 * x[i]
    x[i + 1] = 2 * x[i] - x[i - 1] + a * dt**2

steady = x[int(0.75 * n):]
amplification = float(np.max(np.abs(steady)) * wn**2)

json.dump({"ratio": w / wn, "amplification": amplification}, open("result.json", "w"))
print(f"amplification at resonance: {amplification:.3f}")
"""

(input_dir / "oscillator.py").write_text(script)
print(f"Wrote {input_dir}/oscillator.py")

Wrote /Users/krishna/dev/DesignSafe/Dapi-Tapis/dapi/examples/python/oscillator_job/oscillator.py


## Point Tapis at the folder

Jobs read inputs from DesignSafe storage, not from your disk, so the folder needs a `tapis://` address. On DesignSafe JupyterHub the folder already lives on that storage and translates directly; from anywhere else, dapi uploads it once.

Docs: [Path translation](https://designsafe-ci.github.io/dapi/files#path-translation).

In [3]:
try:
    input_uri = ds.files.to_uri(str(input_dir))  # on DesignSafe JupyterHub
except ValueError:
    prep = ds.jobs.prepare_inputs("python-s3", str(input_dir))
    input_uri = ds.files.to_uri(prep["staged_dir"])
print("Input URI:", input_uri)

Uploading staged files to tapis://designsafe.storage.default/kks32/dapi-staging/oscillator_job (Tapis files API)


Uploaded 1 file(s); staged_dir is tapis://designsafe.storage.default/kks32/dapi-staging/oscillator_job


Input URI: tapis://designsafe.storage.default/kks32/dapi-staging/oscillator_job


## Generate the job request

`generate()` reads the `python-s3` app definition and builds the complete request, resources, queue, the script to run, so you never write the JSON by hand. Print it; this is exactly what Tapis will receive.

Docs: [Job Submission](https://designsafe-ci.github.io/dapi/jobs#job-submission) and the [`generate()` parameters](https://designsafe-ci.github.io/dapi/jobs#generate-parameters).

In [4]:
import json

allocation = "DS-Portal-SPARC2026"  # <-- replace with your allocation

job = ds.jobs.generate(
    app_id="python-s3",
    input_dir_uri=input_uri,
    script_filename="oscillator.py",
    node_count=1,
    cores_per_node=1,
    max_minutes=10,
    queue="skx-dev",
    allocation=allocation,
)
job["name"] = "first-dapi-job"
print(json.dumps(job, indent=2, default=str))

{
  "name": "first-dapi-job",
  "appId": "python-s3",
  "appVersion": "1.0.0",
  "description": "General-purpose execution on Stampede3 \u2014 Python by default, any binary via BINARY \u2014 with optional PyLauncher and MPI support, pre/post scripts, and a machine-readable job-summary.json for provenance.",
  "execSystemId": "stampede3",
  "archiveSystemId": "designsafe.storage.default",
  "archiveSystemDir": "${EffectiveUserId}/tapis-jobs-archive/${JobCreateDate}/${JobName}-${JobUUID}",
  "archiveOnAppError": true,
  "execSystemLogicalQueue": "skx-dev",
  "nodeCount": 1,
  "coresPerNode": 1,
  "maxMinutes": 10,
  "memoryMB": 192000,
  "isMpi": false,
  "tags": [],
  "fileInputs": [
    {
      "name": "Input Directory",
      "sourceUrl": "tapis://designsafe.storage.default/kks32/dapi-staging/oscillator_job",
      "autoMountLocal": true,
      "targetPath": "inputDirectory"
    }
  ],
  "parameterSet": {
    "appArgs": [
      {
        "name": "Input Script",
        "arg": "oscilla

## Submit and monitor

`submit()` hands the request to Tapis and returns a job object; `monitor()` polls until the job reaches a terminal state, printing each stage as it passes, staging, queueing, running, archiving.

Docs: [Job Monitoring](https://designsafe-ci.github.io/dapi/jobs#job-monitoring).

In [5]:
submitted = ds.jobs.submit(job)
print("Job UUID:", submitted.uuid)

final = submitted.monitor(interval=15)
ds.jobs.interpret_status(final, submitted.uuid)

Job submitted successfully. UUID: bbccb9b2-2172-463b-addf-2e19f63d497e-007


Job UUID: bbccb9b2-2172-463b-addf-2e19f63d497e-007

Monitoring Job: bbccb9b2-2172-463b-addf-2e19f63d497e-007


Waiting for job to start: 0 checks [00:00, ? checks/s]

Waiting for job to start: 0 checks [00:00, ? checks/s, Status: PENDING]

Waiting for job to start: 1 checks [00:15, 15.11s/ checks, Status: PENDING]

...

Monitoring job (Status: ARCHIVING):  18%|█▊        | 7/40 [01:30<07:51, 14.29s/ checks]

Monitoring job (Status: ARCHIVING):  18%|█▊        | 7/40 [01:45<07:51, 14.29s/ checks]

Monitoring job (Status: ARCHIVING): 100%|██████████| 40/40 [01:45<00:00, 14.29s/ checks]

Monitoring job (Status: ARCHIVING): 100%|██████████| 40/40 [01:45<00:00,  2.64s/ checks]

	Status: FINISHED
Job bbccb9b2-2172-463b-addf-2e19f63d497e-007 completed successfully.


## Where the time went

The runtime summary splits the wall clock into the stages the monitor showed. For a ten-second calculation, almost everything is staging and queueing, which is why real studies batch many runs per job.

Docs: [Runtime Summary](https://designsafe-ci.github.io/dapi/jobs#runtime-summary).

In [6]:
submitted.print_runtime_summary()


Runtime Summary
---------------


QUEUED  time: 00:00:01
RUNNING time: 00:00:46
TOTAL   time: 00:02:56
---------------


## The archive

Tapis copied everything the job produced to your storage. `tapisjob.out` holds the script's stdout, and the input folder came back with `result.json` written beside the script.

Docs: [Output Management](https://designsafe-ci.github.io/dapi/jobs#output-management).

In [7]:
archive_uri = submitted.archive_uri
print("Archive:", archive_uri)
for it in ds.files.list(archive_uri + "/inputDirectory"):
    print(" ", it.type, it.name)

Archive: tapis://designsafe.storage.default/kks32/tapis-jobs-archive/2026-08-11Z/first-dapi-job-bbccb9b2-2172-463b-addf-2e19f63d497e-007


  file oscillator.py
  file result.json


## The result

In [8]:
result = json.loads(submitted.get_output_content("inputDirectory/result.json"))
print(json.dumps(result, indent=2))
assert 9 < result["amplification"] < 11
print(f"\nD at resonance = {result['amplification']:.2f}, close to 1/(2*0.05) = 10")

{
  "ratio": 1.0,
  "amplification": 10.00000162793735
}

D at resonance = 10.00, close to 1/(2*0.05) = 10


## Where to go next

Change `w = 1.0 * wn` to another ratio and resubmit; you are computing other points of the curve one job at a time. The [PyLauncher sweep](../pylauncher/pylauncher_sweep.ipynb) runs all 25 points inside a single job, and the [workflows examples](https://designsafe-ci.github.io/dapi/workflows) chain jobs that feed each other.